# Training Grover's circuit on a simulated noisy chip (digital-qpu v0.16.0)

The fixed 3-qubit Grover circuit finds the marked item (101) with **94.5%** on a perfect machine but only about **37%** on the noisy `dq-5` chip.
Here its single-qubit layers become **trainable angles** (the oracle and the CCZ gates stay fixed), trained by the parameter-shift rule.

**Targets (set before running, EXPERIMENTS.md v0.16.0)**
1. Ideal chip: best trained circuit >= 0.99
2. `dq-5` (training calibration): best trained circuit >= 0.45
3. 10 calibration days never seen in training: trained minus fixed Grover >= +0.05 on average

**Honest limits:** the reward needs the marked item, so this learns the best *circuit* for a known task on a simulated chip; it is not a better search, and it runs on a classical simulator (no quantum speedup).

Run the cells in order. Total time is roughly 10-30 minutes.

In [ ]:
!pip install -q "git+https://github.com/akosidave31/digital-qpu@v0.16.0"
import digital_qpu
print("digital-qpu", digital_qpu.__version__)

## 1. Sanity check: Grover is the starting point of the trainable circuit

In [ ]:
import time
from digital_qpu import DEVICES
from digital_qpu.variational import VariationalGrover, fixed_grover_success, experiment, summarize
for name in ("ideal", "dq-5"):
    dev = DEVICES[name]
    for rounds in (2, 1):
        vg = VariationalGrover(rounds=rounds)
        t0 = time.time()
        s = vg.success(vg.grover_init(), dev)
        print(f"{name:6} rounds={rounds}: trainable circuit at Grover point {s:.4f} | fixed Grover {fixed_grover_success(dev, rounds):.4f} | {time.time()-t0:.2f} s per evaluation")

## 2. Train and test (the whole experiment)
Epoch progress is printed every 5 epochs. If Colab disconnects, rerun this cell.

In [ ]:
t0 = time.time()
out = experiment(epochs=40, lr=0.05, test_days=range(1, 11))
print(f"total {time.time()-t0:.0f} s")

## 3. Verdict against the targets

In [ ]:
for line in summarize(out):
    print(line)

## 4. Training curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.5))
for key, run in out["runs"].items():
    ax.plot(run["history"], label=f"{key} (best {run['best']:.3f})")
ax.axhline(out["runs"]["dq-5/rounds2"]["start"], color="gray", ls="--", lw=1, label="fixed Grover on dq-5")
ax.set_xlabel("epoch"); ax.set_ylabel("P(marked answer)"); ax.set_ylim(0, 1.02); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.show()

## 5. Results to paste back (and a saved copy)

In [ ]:
import json
json.dump(out, open("grover_training_results.json", "w"), indent=1)
print("runs:")
for k, r in out["runs"].items():
    print(f"  {k:22} start {r['start']:.4f}  best {r['best']:.4f}  ({r['seconds']:.0f} s)")
print("unseen days (fixed Grover vs trained):")
for d, row in out["test"].items():
    print(f"  day {d:>2}: " + "  ".join(f"{k} {v:.3f}" for k, v in row.items()))